In [4]:
# Import necessary libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

import swcol as sw

In [5]:
scenario_name = 'JET'
scenario_path = '../../../data/Colombia/scenarios/2_jet/'
model_inputs_path = scenario_path+'inputs/'
model_outputs_path = scenario_path+'outputs/'
years = [2023, 2050]
dema_path = '../../../data/XM-API/variable_query/2022-12-01_2023-11-30/'

In [6]:
sw.scenarios.table(model_outputs_path, model_inputs_path)

Year,2023,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2040,2045,2050
Tech,,,,,,,,,,,,,,,,,,
Biogas,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,1151.00,171.00,0.00
Biomass,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0,0.0,728.37,249.63,0.00
Eolica,20.00,0.0,1239.18,0.0,255.0,450.0,492.0,0.0,0.0,0.0,0.0,0.0,1528.76,0.0,0.0,208.08,2138.00,4525.09
Geothermal,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,150.00,0.0,0.0,0.00,0.00,0.00
Hidro,41.89,0.0,0.00,1200.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.00,0.00
RunOfRiver,3.75,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.00,0.00
Thermal,0.00,52.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.00,0.00
pv_solar,466.60,1079.9,342.74,218.0,128.0,128.0,123.0,109.0,109.0,91.0,79.0,80.0,1371.66,51.0,69.0,0.00,1421.80,3008.40
Total,532.24,1131.9,1581.92,1418.0,383.0,578.0,615.0,109.0,109.0,91.0,79.0,80.0,3150.42,51.0,69.0,2087.45,3980.43,7533.49


In [7]:
# Transform data
sce_sw = pd.read_csv(model_outputs_path+'/dispatch.csv')
sce_sw['year'] = sce_sw['timestamp'].str[:4]
sce_sw['Energy_TWh_typical_yr'] = sce_sw['Energy_GWh_typical_yr'] / 1000
sce_sw.rename(columns={'gen_tech': 'Tech'}, inplace=True)
sce_sw = sce_sw[sce_sw['timestamp'].str.contains('^'+str(years[0])) == False]
# Plot
sw.scenarios.dispatched_generation(sce_sw, 'Generation', 'TWh', 'year', 'Energy_TWh_typical_yr', 'Tech', scenario_name, 'Switch', 4.6)

In [8]:
em_sw = pd.read_csv(model_outputs_path + '/emissions.csv')
em_sw['AnnualEmissions_MtCO2_per_yr'] = em_sw['AnnualEmissions_tCO2_per_yr'] / 1e6
em_sw = em_sw[em_sw['PERIOD'] != years[0]]
sw.scenarios.annual_emmissions(em_sw, 'PERIOD', 'AnnualEmissions_MtCO2_per_yr', scenario_name, 'Switch')

In [9]:
import pandas as pd

cap_inst = pd.read_csv(model_outputs_path + 'BuildGen.csv')
gen_info = pd.read_csv(model_inputs_path + 'gen_info.csv')

# Unir con info de tecnologías
cap_inst = pd.merge(cap_inst, gen_info, left_on='GEN_BLD_YRS_1',
                    right_on='GENERATION_PROJECT', how='inner')
# Filtrar solo plantas activas (gen_max_age > 1)
cap_inst = cap_inst[cap_inst['gen_max_age'] > 1]
# Expandir cada planta en sus años activos
records = []
for _, row in cap_inst.iterrows():
    start_year = row['GEN_BLD_YRS_2']
    end_year = start_year + row['gen_max_age'] - 1
    for year in range(start_year, end_year + 1):
        if year <= years[1]:
            records.append({
                'year': year,
                'gen_tech': row['gen_tech'],
                'BuildGen': row['BuildGen']
            })

# Crear nuevo DataFrame con capacidad activa por año
cap_inst_expanded = pd.DataFrame(records)
# Sumar capacidad instalada activa por año y tecnología
cap_inst_switch = cap_inst_expanded.groupby(['year', 'gen_tech'], as_index=False)['BuildGen'].sum()

cap_inst_switch = cap_inst_switch[
    (cap_inst_switch['year'] >= years[0]) &
    (cap_inst_switch['year'] <= years[1]) &
    (cap_inst_switch['year'] % 5 == 0)
]

cap_inst_switch['BuildGen'] = cap_inst_switch['BuildGen'] / 1000
cap_inst_switch['Tech'] = cap_inst_switch['gen_tech']
sw.scenarios.dispatched_generation(cap_inst_switch, 'Installed Capacity', 'GWh', 'year', 'BuildGen', 'Tech', scenario_name, 'Switch', 2)

In [10]:
cap_inst_2023 = cap_inst[cap_inst['GEN_BLD_YRS_2'] <= 2023].copy()
cap_inst_2023.rename(columns={'gen_tech': 'Tech'}, inplace=True)
cap_inst_2023 = cap_inst_2023.groupby(['Tech']).agg({
    'BuildGen': 'sum'
}).reset_index()
cap_inst_2023['BuildGen'] = cap_inst_2023['BuildGen']

cap_inst = cap_inst[cap_inst['GEN_BLD_YRS_2'] <= 2037].copy()
cap_inst.rename(columns={'gen_tech': 'Tech'}, inplace=True)

sw.scenarios.installed_capacity(cap_inst_2023, cap_inst, '2023', '2035', scenario_name, 'Switch')